# Lean-27 — Cohérence et témoin : de Finetti construit le livre qui paie, vNM légitime l'affine

Deux résultats du lake [`decision_theory_lean`](../../Probas/decision_theory_lean/) (consommé ici, jamais modifié), mis côte à côte, disent quelque chose que ni l'un ni l'autre ne dit seul.

**1. de Finetti — l'obstruction constructive.** Quand un système de prix est incohérent, le formalisme **construit explicitement un Dutch Book** : pas un certificat d'incohérence, une **stratégie qui fait payer** — la liste des paris, et le gain garanti, état du monde par état du monde. C'est la première attestation dans ce dépôt du patron

$$\text{obstruction abstraite} \;\longrightarrow\; \text{témoin exploitable}$$

**2. vNM — où l'affine est canonique.** Les préférences représentables par utilité espérée sont invariantes par **transformation affine positive** de l'utilité. Le théorème dit précisément **dans quelles conditions** une carte entre deux représentations est légitime : sous les axiomes vNM, sur une échelle cardinale — et pas une seconde de plus.

**Ces deux résultats ensemble corrigent un diagnostic que le dépôt s'était fait à lui-même.** L'erreur du premier Čech affine (la carte d'interpolation posée affine d'emblée) n'était pas « l'affine est idiot » — c'était : **l'affine avait été employé sans qu'on ait démontré la structure qui le rend canonique.** vNM est exactement ce genre de démonstration, pour la carte utilité→préférences. La leçon de méthode de ce notebook :

> **Une carte entre représentations doit d'abord justifier sa forme.**

## Les deux théorèmes, là où ils vivent

| Fait | Énoncé | Où |
|---|---|---|
| Incohérence ⟹ Dutch Book | si $q(A)+q(B) \neq q(A\cap B)+q(A\cup B)$, des mises sur les quatre tickets $(A, B, A\cap B, A\cup B)$ rapportent un gain **strictement positif dans chaque état** | `decision_theory_lean/Coherence/DutchBook.lean:64` — `non_additive_implies_dutch_book`, constructif |
| Cohérence ⟹ additivité | si aucun Dutch Book n'existe sur les quatre tickets, alors l'inclusion–exclusion tient | `Coherence/DutchBook.lean:94` — `coherent_on_implies_additive` (contraposée du premier) |
| Stabilité affine vNM | si $u$ représente $P$, toute $a\cdot u+b$ ($a>0$) représente aussi $P$ | `Utility/Representation.lean:163` — `affine_rep_is_rep` |

Le gain du livret de quatre tickets y est défini (`Coherence/DutchBook.lean:42`, `ieGain`) comme $s_A(\mathbb{1}_A - q_A) + s_B(\mathbb{1}_B - q_B) + s_{AB}(\mathbb{1}_{A\cap B} - q_{AB}) + s_{AU}(\mathbb{1}_{A\cup B} - q_{AU})$ — une mise $s$ **positive achète** le ticket (on paie $s\cdot q$, on reçoit $s$ si l'événement se produit), une mise **négative le vend**. Le miroir Python ci-dessous reprend cette définition **mot pour mot**, en arithmétique exacte (`fractions.Fraction`) : ce que le lake prouve pour tout $\Omega$ fini, le notebook le **mesure** sur une instance — certificat sur l'énoncé, mesure sur l'instance, la dette de dérivation reste explicite.

In [1]:
from fractions import Fraction as F
from itertools import combinations, product

OMEGA = ['AB', 'Ab', 'aB', 'ab']  # les 4 etats : dans A et B, A seul, B seul, ni A ni B

def ind(mot, etat):
    """Indicatrice d'un event (parmi A, B, AB, AU) sur un etat."""
    if mot == 'A':  return F(1) if etat[0] == 'A' else F(0)
    if mot == 'B':  return F(1) if etat[1] == 'B' else F(0)
    if mot == 'AB': return F(1) if etat == 'AB' else F(0)
    if mot == 'AU': return F(1) if etat != 'ab' else F(0)
    raise ValueError(mot)

def gain(prix, mises, etat):
    """Miroir exact de ieGain (DutchBook.lean:42) : somme des s_i * (ind_i - q_i)."""
    return sum(mises[m] * (ind(m, etat) - prix[m]) for m in prix)

def est_dutch_book(prix, mises):
    """Miroir de IsIEArbitrage (DutchBook.lean:49) : gain > 0 dans TOUT etat."""
    return all(gain(prix, mises, w) > 0 for w in OMEGA)

print('Moteur charge : 4 etats,', OMEGA)
print('ieGain/IsIEArbitrage portes en arithmetique exacte (Fraction).')

Moteur charge : 4 etats, ['AB', 'Ab', 'aB', 'ab']
ieGain/IsIEArbitrage portes en arithmetique exacte (Fraction).


## Exercice de démonstration 1 — le système incohérent, et le livre qui en sort

Un agent publie ses prix sur quatre événements d'un monde à quatre états :

| Événement | Prix publié |
|---|---|
| $A$ | $q_A = 0{,}30$ |
| $B$ | $q_B = 0{,}40$ |
| $A \cap B$ | $q_{AB} = 0{,}10$ |
| $A \cup B$ | $q_{AU} = 0{,}85$ |

Chaque prix, pris seul, est plausible (entre 0 et 1). L'incohérence est **structurelle** : l'inclusion–exclusion exige $q_A + q_B = q_{AB} + q_{AU}$, et ici $0{,}70 \neq 0{,}95$. Le théorème `non_additive_implies_dutch_book` (`DutchBook.lean:64`) promet qu'un livre existe ; il ne dit pas lequel. La cellule suivante le **construit**.

In [2]:
PRIX_INCOHERENTS = {'A': F(3, 10), 'B': F(4, 10), 'AB': F(1, 10), 'AU': F(85, 100)}

gauche = PRIX_INCOHERENTS['A'] + PRIX_INCOHERENTS['B']
droite = PRIX_INCOHERENTS['AB'] + PRIX_INCOHERENTS['AU']
print(f'Inclusion-exclusion : q(A)+q(B) = {gauche}   q(AB)+q(AU) = {droite}')
print(f'Violation : {gauche} != {droite}  (ecart {droite - gauche})')

# Le livre : acheter A et B, vendre l'intersection ET l'union
MISES = {'A': F(1), 'B': F(1), 'AB': F(-1), 'AU': F(-1)}
cout_net = -(sum(MISES[m] * PRIX_INCOHERENTS[m] for m in PRIX_INCOHERENTS))

print(f'\nMises : A +{MISES["A"]}, B +{MISES["B"]}, AB {MISES["AB"]} (vendu), AU {MISES["AU"]} (vendu)')
print(f'Cout net du livret a l instant 0 : {cout_net} (ventes 19/20 - achats 7/10 : net encaisse 1/4)')
print(f'\n{"etat":5s} {"gain":>8s}   detail par ticket')
for w in OMEGA:
    detail = '  '.join(f'{m}:{MISES[m]}*({ind(m, w)}-{PRIX_INCOHERENTS[m]})={MISES[m]*(ind(m,w)-PRIX_INCOHERENTS[m])}' for m in ['A','B','AB','AU'])
    print(f'{w:5s} {gain(PRIX_INCOHERENTS, MISES, w)!s:>8s}   {detail}')
print(f'\nDutch book (gain > 0 partout) : {est_dutch_book(PRIX_INCOHERENTS, MISES)}')

Inclusion-exclusion : q(A)+q(B) = 7/10   q(AB)+q(AU) = 19/20
Violation : 7/10 != 19/20  (ecart 1/4)

Mises : A +1, B +1, AB -1 (vendu), AU -1 (vendu)
Cout net du livret a l instant 0 : 1/4 (ventes 19/20 - achats 7/10 : net encaisse 1/4)

etat      gain   detail par ticket
AB         1/4   A:1*(1-3/10)=7/10  B:1*(1-2/5)=3/5  AB:-1*(1-1/10)=-9/10  AU:-1*(1-17/20)=-3/20
Ab         1/4   A:1*(1-3/10)=7/10  B:1*(0-2/5)=-2/5  AB:-1*(0-1/10)=1/10  AU:-1*(1-17/20)=-3/20
aB         1/4   A:1*(0-3/10)=-3/10  B:1*(1-2/5)=3/5  AB:-1*(0-1/10)=1/10  AU:-1*(1-17/20)=-3/20
ab         1/4   A:1*(0-3/10)=-3/10  B:1*(0-2/5)=-2/5  AB:-1*(0-1/10)=1/10  AU:-1*(0-17/20)=17/20

Dutch book (gain > 0 partout) : True


### Lecture du témoin : 1/4 dans chaque état, sans risque

Le tableau est la signature d'un **arbitrage** : le gain vaut **exactement $+1/4$ dans chacun des quatre états** — pas « en moyenne », pas « sauf cas pathologique » : *uniformément*. La stratégie se lit d'un coup : on **achète** $A$ et $B$ pour $0{,}30 + 0{,}40 = 0{,}70$, on **vend** $A \cap B$ et $A \cup B$ pour $0{,}10 + 0{,}85 = 0{,}95$ — on encaisse $0{,}95 - 0{,}70 = 0{,}25$ **à l'instant zéro**, et la position combinée est **identiquement nulle à l'échéance** : quels que soient les événements qui se réalisent, ce qu'on paie sur les tickets vendus est exactement couvert par ce qu'on reçoit des tickets achetés (l'indicatrice vérifie $\mathbb{1}_A + \mathbb{1}_B = \mathbb{1}_{A\cap B} + \mathbb{1}_{A\cup B}$, état par état).

C'est cela, un **témoin exploitable** au sens de la Loi I : l'obstruction abstraite (« les prix ne sont pas additifs ») est devenue un **ordre de marché**. Le théorème du lake garantit qu'un tel livre existe pour *toute* violation ; la construction ci-dessus est l'instance — la plus simple : quatre mises de module 1. Le générateur Python l'exhibe ; le certificat Lean garantit qu'il en existerait toujours un.

## Exercice de démonstration 2 — rendre le système cohérent : le témoin disparaît

Réparons **minimalement** : un seul prix bouge. Poser $q_{AU} = 0{,}60$ rétablit $q_A + q_B = 0{,}70 = q_{AB} + q_{AU}$. Le même livret, sur les mêmes états, ne peut plus rien garantir — et plus généralement **aucun** livret ne peut. Pour le voir en Python, on balaie toutes les mises entières de $-12$ à $+12$ (borne $B = 12$) sur les quatre tickets : c'est une **énumération bornée**, pas une preuve générale — la preuve générale est le théorème `coherent_on_implies_additive` (`DutchBook.lean:94`), dont cette énumeration est le sanity check.

In [3]:
PRIX_COHERENTS = dict(PRIX_INCOHERENTS)
PRIX_COHERENTS['AU'] = F(6, 10)

gauche = PRIX_COHERENTS['A'] + PRIX_COHERENTS['B']
droite = PRIX_COHERENTS['AB'] + PRIX_COHERENTS['AU']
print(f'Systeme reprare : q(A)+q(B) = {gauche} = q(AB)+q(AU) = {droite}  -> additif')

# (a) le livret de tout a l'heure, sur le systeme reprare
print(f'\nLivret (1,1,-1,-1) sur le systeme reprare :')
for w in OMEGA:
    print(f'  {w:5s} gain = {gain(PRIX_COHERENTS, MISES, w)}')
print(f'Dutch book ? {est_dutch_book(PRIX_COHERENTS, MISES)}')

# (b) enumeration bornee : aucune mise entiere dans [-12, 12] ne fait un livre
# (prix en dixiemes -> gains scales x10 en entiers, exact)
trouves = []
B = 12
coef = {m: {w: 10 * int(ind(m, w)) - int(PRIX_COHERENTS[m] * 10) for w in OMEGA} for m in PRIX_COHERENTS}
combis = (2 * B + 1) ** 4
for sA, sB, sAB, sAU in product(range(-B, B + 1), repeat=4):
    if (sA, sB, sAB, sAU) == (0, 0, 0, 0):
        continue
    if all(sA * coef['A'][w] + sB * coef['B'][w] + sAB * coef['AB'][w] + sAU * coef['AU'][w] > 0 for w in OMEGA):
        trouves.append((sA, sB, sAB, sAU))
        break
print(f'\nBalayage exhaustif des mises entieres dans [-{B},{B}]^4 ({combis:,} combinaisons, gains x10 exacts en entiers) :')
print(f'Livres trouves : {trouves if trouves else "AUCUN"}')

Systeme reprare : q(A)+q(B) = 7/10 = q(AB)+q(AU) = 7/10  -> additif

Livret (1,1,-1,-1) sur le systeme reprare :
  AB    gain = 0
  Ab    gain = 0
  aB    gain = 0
  ab    gain = 0
Dutch book ? False



Balayage exhaustif des mises entieres dans [-12,12]^4 (390,625 combinaisons, gains x10 exacts en entiers) :
Livres trouves : AUCUN


### Lecture : le témoin disparaît — et ce que le balayage ne prouve pas

Le même livret qui rapportait $1/4$ partout rapporte maintenant **exactement $0$ dans chacun des quatre états** : l'écart de prix a disparu, et la position combinée — neutre dès l'instant zéro — ne devient rien à l'échéance non plus. La machine à gagner est démontée, jusqu'à la dernière pièce. Et le balayage de $390{,}625$ combinaisons de mises ($25^4$) n'en trouve **aucune** qui refasse un livre. Mais il faut dire précisément ce que ce balayage établit : *aucun livre à mises entières bornées*. La propriété générale — *aucun livre du tout* — n'est pas énumérable (les mises réelles sont infinies) ; elle est **démontrée** dans le lake (`DutchBook.lean:94`). C'est la frontière exacte du partage de travail :

| | Générateur Python | Certificat Lean |
|---|---|---|
| Ce qu'il fournit | le livre concret, les montants, l'instance | l'impossibilité générale |
| Portée | une instance de prix | toute violation / toute cohérence |
| Statut | **mesure** | **preuve** |

Confondre les deux colonnes serait exactement l'erreur que ce notebook documente par ailleurs (une carte employée sans justification).

## vNM — l'affine est enfin légitime

Passons du côté des préférences. Le théorème de représentation de von Neumann–Morgenstern relie un ordre sur les loteries à une utilité espérée ; le lemme `affine_rep_is_rep` (`Utility/Representation.lean:163`) en prouve la partie cardinale : si $u$ représente $P$, alors $a \cdot u + b$ ($a > 0$) représente **aussi** $P$ — car $E_p[a\,u+b] = a\,E_p[u] + b$, et une affine croissante préserve l'ordre.

**Ce que le lemme légitime** : changer l'origine ($b$) et l'unité ($a$) d'une échelle d'utilité — température en Celsius ou Fahrenheit, utilité en points ou en dizaines. **Ce qu'il ne légitime pas** : toute autre forme. Le test suivant est une **discrimination expérimentale** : sur les 66 loteries simples d'un simplex à pas $1/10$ (toutes les $(p_1,p_2,p_3)$ entières sur dixièmes), on compare les ordres induits par $u$, par $v = 3u+2$ (affine), et par $w = u^2$ (non affine).

In [4]:
def loteries_simplexe(pas=10):
    """Toutes les lois de probabilite (p1,p2,p3) a pas 1/10."""
    out = []
    for i in range(pas + 1):
        for j in range(pas + 1 - i):
            out.append((F(i, pas), F(j, pas), F(pas - i - j, pas)))
    return out

def esperance(u, loi):
    return sum(p * x for p, x in zip(loi, u))

U = (F(1), F(2), F(4))                 # utilite de depart
V = tuple(3 * x + 2 for x in U)        # affine positive : v = 3u + 2
W = tuple(x * x for x in U)            # non affine : w = u^2

LOT = loteries_simplexe()
print(f'{len(LOT)} loteries sur le simplex a pas 1/10  (u = {U},  v = 3u+2 = {V},  w = u^2 = {W})')

def paires_divergentes(f, g):
    """Paires (p,q) ou l'ordre induit par f differe de celui induit par g."""
    div = []
    for p, q in combinations(LOT, 2):
        ef_p, ef_q = esperance(f, p), esperance(f, q)
        eg_p, eg_q = esperance(g, p), esperance(g, q)
        sf = (ef_p > ef_q) - (ef_p < ef_q)
        sg = (eg_p > eg_q) - (eg_p < eg_q)
        if sf != sg:
            div.append((p, q, ef_p, ef_q, eg_p, eg_q))
    return div

n_paires = len(LOT) * (len(LOT) - 1) // 2
div_affine = paires_divergentes(U, V)
div_carre = paires_divergentes(U, W)
print(f'\nComparaisons : {n_paires} paires ordonnees, deux fois')
print(f'u vs v=3u+2 (affine)  : {len(div_affine)} divergences')
print(f'u vs w=u^2 (non affine): {len(div_carre)} divergences')

66 loteries sur le simplex a pas 1/10  (u = (Fraction(1, 1), Fraction(2, 1), Fraction(4, 1)),  v = 3u+2 = (Fraction(5, 1), Fraction(8, 1), Fraction(14, 1)),  w = u^2 = (Fraction(1, 1), Fraction(4, 1), Fraction(16, 1)))

Comparaisons : 2145 paires ordonnees, deux fois
u vs v=3u+2 (affine)  : 0 divergences
u vs w=u^2 (non affine): 124 divergences


### Lecture de l'invariance : zéro divergence — la carte affine est inoffensive

Sur les $2\,145$ paires de loteries, l'ordre induit par $u$ et celui induit par $v = 3u+2$ **coïncident exactement** : zéro divergence. C'est la mesure de l'invariance que `affine_rep_is_rep` prouve en général — sur cette instance, elle est parfaite. Aucune décision, aucun classement, aucune préférence ne bouge quand on recarde l'échelle d'utilité par une affine positive : la carte $u \mapsto 3u+2$ est **légitime**, et le lemme dit *pourquoi* — l'espérance se transforme en $E_p[3u+2] = 3E_p[u]+2$, et $x \mapsto 3x+2$ est croissante sur $\mathbb{R}$.

Le carré, lui, diverge. Les deux cellules suivantes exhibent **les deux défauts** : une paire **indifférente** sous $u$ mais strictement ordonnée sous $w$ (le carré fabrique des préférences là où il n'y en avait pas), et une paire dont l'ordre **s'inverse** carrément.

In [5]:
# Defaut 1 : indifferents sous u, ordonnes sous w
indiff = []
for p, q in combinations(LOT, 2):
    if esperance(U, p) == esperance(U, q) and esperance(W, p) != esperance(W, q):
        indiff.append((p, q))
# Defaut 2 : ordre inverse
flip = []
for p, q in combinations(LOT, 2):
    eu_p, eu_q = esperance(U, p), esperance(U, q)
    ew_p, ew_q = esperance(W, p), esperance(W, q)
    if (eu_p > eu_q and ew_p < ew_q) or (eu_p < eu_q and ew_p > ew_q):
        flip.append((p, q, eu_p, eu_q, ew_p, ew_q))

p, q = indiff[0]
print(f'Defaut 1 - indifference fabriquee :')
print(f'  p = {tuple(float(x) for x in p)}   E_p[u] = {esperance(U, p)}   E_p[w] = {esperance(W, p)}')
print(f'  q = {tuple(float(x) for x in q)}   E_q[u] = {esperance(U, q)}   E_q[w] = {esperance(W, q)}')
print(f'  sous u : INDIFFERENT ({esperance(U, p)} = {esperance(U, q)}) ; sous w : preferer q de {esperance(W, q) - esperance(W, p)}')

p, q, eu_p, eu_q, ew_p, ew_q = flip[0]
print(f'\nDefaut 2 - ordre inverse :')
print(f'  p = {tuple(float(x) for x in p)}   E_p[u] = {eu_p}   E_p[w] = {ew_p}')
print(f'  q = {tuple(float(x) for x in q)}   E_q[u] = {eu_q}   E_q[w] = {ew_q}')
print(f"  sous u : p > q ({eu_p} > {eu_q}) ; sous w : p < q ({ew_p} < {ew_q})  -> le classement S'INVERSE")

Defaut 1 - indifference fabriquee :
  p = (0.0, 0.3, 0.7)   E_p[u] = 17/5   E_p[w] = 62/5
  q = (0.2, 0.0, 0.8)   E_q[u] = 17/5   E_q[w] = 13
  sous u : INDIFFERENT (17/5 = 17/5) ; sous w : preferer q de 3/5

Defaut 2 - ordre inverse :
  p = (0.0, 0.4, 0.6)   E_p[u] = 16/5   E_p[w] = 56/5
  q = (0.3, 0.0, 0.7)   E_q[u] = 31/10   E_q[w] = 23/2
  sous u : p > q (16/5 > 31/10) ; sous w : p < q (56/5 < 23/2)  -> le classement S'INVERSE


### Lecture du contre-exemple : le carré n'est pas une échelle, c'est une loupe

La paire du défaut 1 dit tout : deux loteries **exactement indifférentes** sous $u$ deviennent strictement ordonnées sous $w = u^2$. Le mécanisme est la convexité : $u \mapsto u^2$ écrase les petits écarts et **amplifie les grands** (les utilités $1, 2, 4$ deviennent $1, 4, 16$ — le rapport entre les extrêmes passe de 4 à 16). Une loterie qui prend des valeurs extrêmes avec petite probabilité, indifférente face à une loterie concentrée sous $u$, devient *strictement préférée* sous $w$ : le carré fabrique du goût pour le risque (ou l'aversion, selon le côté) que l'agent n'a pas. C'est précisément pourquoi vNM ne garantit l'invariance **que** pour l'affine positive : toute forme non linéaire encode des préférences supplémentaires — elle n'est pas un changement d'échelle, c'est un **autre agent**.

D'où la leçon de méthode, qui referme la boucle ouverte en introduction. La carte affine du premier Čech n'était pas absurde — elle était **non justifiée** : posée avant qu'on ait démontré la structure (symétries du problème, axiomes d'invariance) qui la rend canonique. `affine_rep_is_rep` est le modèle de ce que « justifier la forme » veut dire : un théorème, pas un réflexe. **Toute carte entre représentations doit d'abord justifier sa forme** — l'affine quand une invariance l'exige, et autre chose quand une autre structure l'exige.

## Résumé — deux théorèmes, un patron

| | de Finetti | vNM |
|---|---|---|
| Obstruction | prix non additifs ($0{,}70 \neq 0{,}95$) | carte non justifiée (Czech affine) |
| Témoin | un livret : $(+1,+1,-1,-1)$, gain $+1/4$ **uniforme** | deux loteries indifférentes, ordonnées sous $u^2$ |
| Disparition | $q_{AU} := 0{,}60$ ⟹ balayage intégral : aucun livre | $v = 3u+2$ ⟹ $2\,145$ paires, **0** divergence |
| Certificat | `DutchBook.lean:64` / `:94` | `Representation.lean:163` |
| Leçon | l'incohérence **coûte**, littéralement | la carte doit **justifier sa forme** |

Les deux colonnes sont les deux faces d'une même loi — l'obstruction abstraite devient un témoin exploitable. C'est l'attestation indépendante nº 1 de la Loi I dans ce dépôt (l'autre : Brown-Sandholm, safe subgame solving, côté GameTheory) ; deux lakes distincts, même patron : c'est ce qui autorise à en faire une **loi** plutôt qu'une intuition.

**Générateur ≠ certificat**, une dernière fois : tout ce que ce notebook calcule est une *mesure sur instance* (prix donnés, loteries du simplex à pas 1/10) ; tout ce que le lake énonce vaut pour *tout* $\Omega$ fini et *toute* préférence représentable. La dette de dérivation est ici nulle côté de Finetti (le miroir suit `ieGain` définition pour définition) et assumée côté vNM (le sens *existence* de la représentation — Herstein–Milnor — est un jalon ouvert documenté dans le lake, pas une preuve).

### La correction d'un diagnostic antérieur

Les deux démonstrations referment la correction annoncée en ouverture. L'erreur du premier Čech affine ([ICT-15d](../../IIT/ICT-Series/ICT-15j-NerveDiscriminant.ipynb)) ne se corrige pas en condamnant la carte — elle se corrige en produisant, chaque fois, la structure qui légitime la forme employée. Écrite comme le lit un carnet de bord :

```
ERREUR INITIALE   :  on transporte une mesure avec une carte affine, sans axiome.
DIAGNOSTIC       :  « l'affine produit des artefacts numériques ».
CORRECTION       :  axiome vNM d'abord, puis l'affine est légitime.
                    OU axiome manquant : l'affine N'EST PAS canonique.
                    Le témoin de Finetti est la sanction constructive du cas incohérent.
```

Le diagnostic initial était **mal posé** : il blâmait la carte au lieu de blâmer l'absence de justification. Et la sanction, quand la structure manque, n'est pas un verdict abstrait — le témoin de Finetti est un **livre qui paie** : l'obstruction devenue objet opérationnel.

C'est la leçon de méthode, une dernière fois : **une carte entre représentations doit d'abord justifier sa forme.**

## Exercices

### Exercice 1 — Votre propre incohérence

Construisez un système de prix **différent** (quatre prix dans $[0,1]$, violation non nulle de l'inclusion–exclusion) et faites **sortir votre propre Dutch Book** : trouvez des mises (elles n'ont pas besoin d'être entières — pensez à suivre l'écart $q_{AB}+q_{AU} - q_A - q_B$) dont le gain est strictement positif dans les quatre états. Vérifiez avec `est_dutch_book`, et affichez le tableau état × gain.

In [6]:
# Exercice 1 a completer : vos prix, vos mises, votre tableau de gains.
# mon_prix = {'A': ..., 'B': ..., 'AB': ..., 'AU': ...}
# mes_mises = {'A': ..., 'B': ..., 'AB': ..., 'AU': ...}
# result_exo1 = None  # TODO etudiant : (mon_prix, mes_mises, gain_par_etat)
result_exo1 = None  # TODO etudiant
print('Exercice 1 a completer : systeme incoherent + livre qui paie partout.')

Exercice 1 a completer : systeme incoherent + livre qui paie partout.


### Exercice 2 — L'autre classe d'incohérence : le prix impossible

L'inclusion–exclusion n'est pas la seule façon d'être incohérent. Le lake traite aussi (`Coherence/Probability.lean:51`, `IsSingleDutchBook`) le cas d'un **prix hors bornes** : si $q(A) > 1$, vendre le ticket $A$ est un gain garanti (on encaisse plus que ce qu'on devra jamais payer) ; si $q(A) < 0$, l'acheter l'est. Construisez un système où chaque prix, **pris isolément**, est dans $[0,1]$, l'inclusion–exclusion tient, mais... une **paire** de tickets cache quand même un livre (indice : $q(A) + q(\bar A) \neq 1$ sur le couple complémentaire — une cinquième forme de violation que notre miroir à quatre tickets ne couvre pas). Dites en une phrase ce qu'il faudrait ajouter au moteur pour l'attraper.

In [7]:
# Exercice 2 a completer : construire le systeme piege, montrer le livre cache,
# et decrire l'extension du moteur.
# result_exo2 = None          # TODO etudiant : (prix_pieges, mises, explication)
# extension_moteur = ""       # TODO etudiant
result_exo2 = None  # TODO etudiant
extension_moteur = ""  # TODO etudiant
print('Exercice 2 a completer : la paire complementaire nonnormalizee.')

Exercice 2 a completer : la paire complementaire nonnormalizee.


### Exercice 3 — Une autre forme non affine : l'exponentielle

Remplacez $w = u^2$ par $w = e^{u}$ (utilités $e, e^2, e^4$). Les deux défauts (indifférence fabriquée, ordre inversé) réapparaissent-ils ? Sur le même simplex à pas $1/10$, comptez les divergences $u$ vs $e^u$ et comparez au compte du carré — la loupe convexe est-elle plus ou moins déformante ? Concluez en une phrase : *le compte de divergences mesure-t-il la « non-affinité » d'une transformation, et quelle propriété des transformations affines le compte à zéro capturing exactement ?*

In [8]:
# Exercice 3 a completer : w = exp(u), compter les divergences u vs exp(u),
# exhiber le premier flip, conclure sur le compte comme mesure de non-affinite.
# import math
# EXP_U = tuple(F(math.exp(x)).limit_denominator(10**9) for x in U)  # approx rationnelle
# result_exo3 = None   # TODO etudiant : (nb_divergences, exemple_flip, conclusion)
result_exo3 = None  # TODO etudiant
print('Exercice 3 a completer : divergences de exp(u) vs u sur le simplex.')

Exercice 3 a completer : divergences de exp(u) vs u sur le simplex.
